# Multi-Appliance Recognition from High-Frequency Signals

Companion notebook for Chapter 3. Builds the Fryze-current-decomposition and
distance-similarity-matrix features on real, multi-label PLAID data, then
trains a real multi-label CNN classifier to compare them. See the chapter
text for the full narrative and citations; this notebook is the code and
results behind it.

Data comes from `resources/nilm-code/MLCFCD` (the author's own research
code, gitignored in this repo). Re-running this notebook end to end requires
that directory to be present locally.

In [ ]:
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from lets_plot import (
    LetsPlot,
    aes,
    coord_flip,
    element_blank,
    facet_wrap,
    geom_bar,
    geom_jitter,
    geom_line,
    geom_point,
    geom_text,
    ggplot,
    ggsize,
    labs,
    scale_color_manual,
    scale_fill_manual,
    theme,
)
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
import torch
import torch.nn as nn

from ark.plot.theme import modern_theme
from ark.plot.tokens import BLUES_SEQUENTIAL, BRAND_PALETTE, GRAY_400

LetsPlot.setup_html()

In [ ]:
def paa(series: np.ndarray, w: int) -> np.ndarray:
    """Reduce a 1D signal to `w` points via Piecewise Aggregate Approximation.

    Same building block Chapter 2 used: splits `series` into `w` equal
    segments and replaces each with its mean.

    Args:
        series: 1D array of length T_s (the raw activation window).
        w: Target length after reduction (the embedding size).

    Returns:
        1D array of length `w`.
    """
    n = len(series)
    bounds = np.linspace(0, n, w + 1).astype(int)
    return np.array([series[bounds[k] : bounds[k + 1]].mean() for k in range(w)])


def distance_matrix(x: np.ndarray) -> np.ndarray:
    """Build the pairwise absolute-distance matrix D of a 1D signal.

    Args:
        x: 1D array of length w (already PAA-reduced).

    Returns:
        w x w array where entry [i, j] is |x[i] - x[j]|.
    """
    return np.abs(x[:, None] - x[None, :])


def fryze_power_decomposition(i: np.ndarray, v: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Split an activation current into its active and non-active components.

    The active component is the purely resistive current in phase with the
    voltage, carrying the same active power as the real signal; the
    non-active component is whatever current is left over.

    Args:
        i: 1D activation current window.
        v: 1D activation voltage window, same length as `i`.

    Returns:
        Tuple of (active current, non-active current), each the same shape
        as `i`.
    """
    active_power = (i * v).mean()
    v_rms_sq = (v**2).mean()
    i_active = active_power * v / v_rms_sq
    i_non_active = i - i_active
    return i_active, i_non_active


def vi_image(current: np.ndarray, voltage: np.ndarray, w: int) -> np.ndarray:
    """Rasterize a current/voltage trajectory into a binary w x w V-I image.

    Centers and bins the current and voltage onto a w x w grid, marking a
    cell wherever the trajectory passes through it, then normalizes so the
    busiest cell is 1. This is the baseline feature representation the
    decomposed-current features are compared against.

    Args:
        current: 1D array of raw activation current.
        voltage: 1D array of raw activation voltage, same length.
        w: Grid resolution.

    Returns:
        w x w array, values in [0, 1].
    """
    c = np.clip(current, -np.abs(current).max(), np.abs(current).max())
    v = np.clip(voltage, -np.abs(voltage).max(), np.abs(voltage).max())
    dc = (c.max() - c.min()) / (w - 1)
    dv = (v.max() - v.min()) / (w - 1)
    ci = np.clip(((c - c.min()) / dc).astype(int), 0, w - 1)
    vi = np.clip(((v - v.min()) / dv).astype(int), 0, w - 1)
    img = np.zeros((w, w))
    for a, b in zip(ci, vi, strict=False):
        img[a, w - b - 1] += 1
    return img / img.max()


def normalize(x: np.ndarray) -> np.ndarray:
    """Scale a signal by its own peak absolute value.

    Args:
        x: 1D array.

    Returns:
        1D array with the same shape, max absolute value 1.
    """
    return x / (np.abs(x).max() + 1e-8)

In [ ]:
class Conv1DEncoder(nn.Module):
    """Four-stage 1D convolutional encoder for signal-shaped features.

    Matches the author's own `MLCFCD` architecture: four conv layers
    (16/32/64/128 feature maps, kernel sizes 5/5/3/3, stride 2), each
    followed by ReLU, then adaptive average pooling down to one value per
    channel. Used for the "current" and "decomposed current" features,
    which are 1D signals, not images.

    Args:
        in_channels: Number of input channels (1 for raw current, 2 for
            active + non-active decomposed current).
    """

    def __init__(self, in_channels: int):
        super().__init__()
        channels = (16, 32, 64, 128)
        kernels = (5, 5, 3, 3)
        layers = []
        c_in = in_channels
        for c_out, k in zip(channels, kernels, strict=True):
            layers += [nn.Conv1d(c_in, c_out, k, stride=2), nn.ReLU()]
            c_in = c_out
        self.conv = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Encode a batch of 1D features into a 128-dim vector each.

        Args:
            x: Tensor of shape (batch, in_channels, w).

        Returns:
            Tensor of shape (batch, 128).
        """
        x = self.conv(x)
        return nn.functional.adaptive_avg_pool1d(x, 1).flatten(1)


class Conv2DEncoder(nn.Module):
    """Four-stage 2D convolutional encoder for image-shaped features.

    Same channel/kernel/stride progression as `Conv1DEncoder`, but for 2D
    inputs: the V-I image and the decomposed distance-similarity matrices.

    Args:
        in_channels: Number of input channels (1 for the V-I image, 2 for
            the active + non-active distance matrices).
    """

    def __init__(self, in_channels: int):
        super().__init__()
        channels = (16, 32, 64, 128)
        kernels = (5, 5, 3, 3)
        layers = []
        c_in = in_channels
        for c_out, k in zip(channels, kernels, strict=True):
            layers += [nn.Conv2d(c_in, c_out, k, stride=2), nn.ReLU()]
            c_in = c_out
        self.conv = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Encode a batch of 2D features into a 128-dim vector each.

        Args:
            x: Tensor of shape (batch, in_channels, w, w).

        Returns:
            Tensor of shape (batch, 128).
        """
        x = self.conv(x)
        return nn.functional.adaptive_avg_pool2d(x, 1).flatten(1)


class MultiLabelCNN(nn.Module):
    """CNN multi-label appliance classifier with a per-appliance softmax head.

    Wraps either encoder with the same FC head (128-512-1024-128) used in
    the author's own multi-label model, then a final layer producing two
    logits per appliance (off, on) instead of one. Reshaping to
    (batch, 2, n_appliances) and training with cross-entropy is
    mathematically a reparametrized sigmoid: softmax over two logits
    `(a, b)` is `sigmoid(a - b)`. The practical difference is that the
    decision boundary (which of the two logits wins) is learned end to end,
    rather than an external 0.5 cutoff applied to a probability afterward.

    Args:
        encoder: A `Conv1DEncoder` or `Conv2DEncoder` instance.
        n_appliances: Number of appliance classes.
        dropout: Dropout probability before the final layer.
    """

    def __init__(self, encoder: nn.Module, n_appliances: int, dropout: float = 0.25):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Sequential(
            nn.Linear(128, 512),
            nn.ReLU(),
            nn.Linear(512, 1024),
            nn.ReLU(),
            nn.Linear(1024, 128),
            nn.ReLU(),
        )
        self.dropout = nn.Dropout(dropout)
        self.fc_out = nn.Linear(128, n_appliances * 2)
        self.n_appliances = n_appliances

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Predict per-appliance (off, on) logits for a batch of features.

        Args:
            x: A batch of features matching the encoder's expected shape.

        Returns:
            Tensor of shape (batch, 2, n_appliances).
        """
        z = self.encoder(x)
        z = self.head(z)
        z = self.dropout(z)
        out = self.fc_out(z)
        return out.view(-1, 2, self.n_appliances)


def train_and_evaluate(
    features: np.ndarray,
    y: np.ndarray,
    encoder: nn.Module,
    train_idx: np.ndarray,
    test_idx: np.ndarray,
    epochs: int = 300,
    lr: float = 1e-3,
) -> dict:
    """Train a `MultiLabelCNN` on one feature representation and evaluate it.

    Args:
        features: Array of shape (n_samples, channels, ...) matching the
            encoder's expected input shape.
        y: Multi-hot appliance-state matrix, shape (n_samples, n_appliances).
        encoder: A fresh, untrained `Conv1DEncoder` or `Conv2DEncoder`.
        train_idx: Row indices for the training split.
        test_idx: Row indices for the held-out split.
        epochs: Number of full-batch gradient steps to train for.
        lr: Adam learning rate.

    Returns:
        Dict with example-based F1 (`f1_eb`), macro F1 (`f1_macro`), the
        held-out `y_test`/`y_pred` multi-hot arrays (for per-appliance and
        error-type analysis).
    """
    x_train = torch.tensor(features[train_idx]).float()
    x_test = torch.tensor(features[test_idx]).float()
    y_train = torch.tensor(y[train_idx]).long()
    y_test = y[test_idx]

    torch.manual_seed(0)
    model = MultiLabelCNN(encoder, n_appliances=y.shape[1])
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    model.train()
    for _ in range(epochs):
        optimizer.zero_grad()
        loss = loss_fn(model(x_train), y_train)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        y_pred = model(x_test).argmax(dim=1).numpy()

    f1_eb = f1_score(y_test, y_pred, average="samples", zero_division=0)
    f1_macro = f1_score(y_test, y_pred, average="macro", zero_division=0)
    return {"f1_eb": f1_eb, "f1_macro": f1_macro, "y_test": y_test, "y_pred": y_pred}

## Loading a real multi-appliance activation window

Loads the real PLAID multi-label dataset: 1154 aggregated activation events,
each labeled with every appliance active in the background, not just the one
that changed state. Picks out one real event with more than one appliance
running to make the problem concrete before building any features.

In [ ]:
plaid_dir = "../../resources/nilm-code/MLCFCD/data/plaid/"
current = np.load(plaid_dir + "current.npy", allow_pickle=True)
voltage = np.load(plaid_dir + "voltage.npy", allow_pickle=True)
labels = np.load(plaid_dir + "labels.npy", allow_pickle=True)

classes = sorted({appliance for active in labels for appliance in active})
class_to_idx = {name: idx for idx, name in enumerate(classes)}

y = np.zeros((len(labels), len(classes)), dtype=np.float32)
for row, active in enumerate(labels):
    for appliance in active:
        y[row, class_to_idx[appliance]] = 1

n_active = y.sum(axis=1)
example = np.where(n_active == 2)[0][0]

example_df = pd.DataFrame({"sample": np.arange(len(current[example])), "current": current[example].astype(np.float64)})

example_plot = (
    ggplot(example_df, aes("sample", "current"))
    + geom_line(color=BRAND_PALETTE[0], size=0.9)
    + labs(
        title="Two appliances, one window",
        subtitle=f"Real PLAID event: {', '.join(labels[example])} both active",
        x="Sample",
        y="Current (A)",
    )
    + modern_theme()
    + ggsize(650, 320)
)
example_plot

## Does the same feature carry over as-is?

Before reaching for anything new, it is worth checking whether the two
representations Chapter 2 already built, the binary V-I image and the raw
activation current, still work once the "current" in question might be the
sum of two or three appliances instead of one. A single 75/25 split is used
throughout, stratified across appliance *combinations*, not just individual
appliances, so a rare triple-activation sample is not stranded entirely in
one side of the split.

In [ ]:
splitter = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(np.zeros(len(y)), y))

w = 50
n_samples = len(current)
vi_features = np.zeros((n_samples, 1, w, w), dtype=np.float32)
current_features = np.zeros((n_samples, 1, w), dtype=np.float32)

for idx in range(n_samples):
    i_raw = current[idx].astype(np.float64)
    v_raw = voltage[idx].astype(np.float64)
    vi_features[idx, 0] = vi_image(i_raw, v_raw, w)
    current_features[idx, 0] = normalize(paa(i_raw, w))

vi_result = train_and_evaluate(vi_features, y, Conv2DEncoder(1), train_idx, test_idx)
current_result = train_and_evaluate(current_features, y, Conv1DEncoder(1), train_idx, test_idx)

comparison_df = pd.DataFrame(
    {
        "representation": ["V-I image", "Activation current"],
        "f1_macro": [round(vi_result["f1_macro"] * 100, 1), round(current_result["f1_macro"] * 100, 1)],
    }
)

comparison_plot = (
    ggplot(comparison_df, aes("representation", "f1_macro", fill="representation"))
    + geom_bar(stat="identity", width=0.5)
    + geom_text(aes(label="f1_macro"), format="{.1f}", va="bottom", nudge_y=1.5, size=8)
    + scale_fill_manual(values=[BLUES_SEQUENTIAL[2], BRAND_PALETTE[0]])
    + labs(
        title="A close call, not a clean win",
        subtitle="Macro F1, PLAID multi-label, held-out 25%",
        x="",
        y="Macro F1 (%)",
    )
    + modern_theme(legend_pos="none")
    + ggsize(500, 340)
)
comparison_plot

## Fryze power decomposition

Splits the same real activation current into an active component (purely
resistive, in phase with the voltage) and a non-active component (whatever
current is left over once the active part is removed). Shown on the same
two-appliance event as the opening figure.

In [ ]:
i_active, i_non_active = fryze_power_decomposition(
    current[example].astype(np.float64), voltage[example].astype(np.float64)
)

decomposed_df = pd.concat(
    [
        pd.DataFrame(
            {
                "sample": np.arange(len(current[example])),
                "current": current[example].astype(np.float64),
                "component": "Source current",
            }
        ),
        pd.DataFrame({"sample": np.arange(len(i_active)), "current": i_active, "component": "Active"}),
        pd.DataFrame(
            {
                "sample": np.arange(len(i_non_active)),
                "current": i_non_active,
                "component": "Non-active",
            }
        ),
    ]
)
decomposed_df["component"] = pd.Categorical(
    decomposed_df["component"], categories=["Source current", "Active", "Non-active"], ordered=True
)

decomposed_plot = (
    ggplot(decomposed_df, aes("sample", "current", color="component"))
    + geom_line(size=0.8)
    + facet_wrap("component", ncol=3, scales="free_y")
    + scale_color_manual(values=BRAND_PALETTE)
    + labs(
        title="One current, two physical components",
        subtitle="Fryze decomposition, same two-appliance event",
        x="Sample",
        y="Current (A)",
    )
    + modern_theme(legend_pos="none")
    + ggsize(650, 320)
)
decomposed_plot

## From decomposed current to a distance matrix

The same `distance_matrix()` building block from Chapter 2, now applied to
the active and non-active components separately and stacked as two
channels, the same way Chapter 2 stacked three phases. Four representations
are compared in total: the V-I image and raw current from above, plus the
decomposed current itself and its distance-similarity matrix.

In [ ]:
decomposed_current_features = np.zeros((n_samples, 2, w), dtype=np.float32)
decomposed_distance_features = np.zeros((n_samples, 2, w, w), dtype=np.float32)

for idx in range(n_samples):
    i_raw = current[idx].astype(np.float64)
    v_raw = voltage[idx].astype(np.float64)
    i_active, i_non_active = fryze_power_decomposition(i_raw, v_raw)
    active_paa = normalize(paa(i_active, w))
    non_active_paa = normalize(paa(i_non_active, w))
    decomposed_current_features[idx, 0] = active_paa
    decomposed_current_features[idx, 1] = non_active_paa
    decomposed_distance_features[idx, 0] = distance_matrix(active_paa)
    decomposed_distance_features[idx, 1] = distance_matrix(non_active_paa)

decomposed_current_result = train_and_evaluate(decomposed_current_features, y, Conv1DEncoder(2), train_idx, test_idx)
decomposed_distance_result = train_and_evaluate(decomposed_distance_features, y, Conv2DEncoder(2), train_idx, test_idx)

full_comparison_df = pd.DataFrame(
    {
        "representation": ["V-I image", "Activation current", "Decomposed current", "Decomposed distance"],
        "f1_macro": [
            round(vi_result["f1_macro"] * 100, 1),
            round(current_result["f1_macro"] * 100, 1),
            round(decomposed_current_result["f1_macro"] * 100, 1),
            round(decomposed_distance_result["f1_macro"] * 100, 1),
        ],
    }
)
full_comparison_df["representation"] = pd.Categorical(
    full_comparison_df["representation"],
    categories=["V-I image", "Activation current", "Decomposed current", "Decomposed distance"],
    ordered=True,
)

full_comparison_plot = (
    ggplot(full_comparison_df, aes("representation", "f1_macro", fill="representation"))
    + geom_bar(stat="identity", width=0.6)
    + geom_text(aes(label="f1_macro"), format="{.1f}", va="bottom", nudge_y=1.5, size=8)
    + scale_fill_manual(values=[BLUES_SEQUENTIAL[1], BLUES_SEQUENTIAL[3], BLUES_SEQUENTIAL[5], BRAND_PALETTE[0]])
    + labs(
        title="Decomposing the current is what actually helps",
        subtitle="Macro F1, PLAID multi-label, held-out 25%",
        x="",
        y="Macro F1 (%)",
    )
    + modern_theme(legend_pos="none", x_axis_angle=15)
    + ggsize(650, 380)
)
full_comparison_plot

## Which appliances specifically?

Macro F1 hides which appliances the winning feature still struggles with.
Per-appliance F1, V-I image against the decomposed distance matrix.

In [ ]:
f1_vi_per_class = f1_score(vi_result["y_test"], vi_result["y_pred"], average=None, zero_division=0) * 100
f1_decomposed_per_class = (
    f1_score(decomposed_distance_result["y_test"], decomposed_distance_result["y_pred"], average=None, zero_division=0)
    * 100
)

per_class_df = pd.concat(
    [
        pd.DataFrame({"appliance": classes, "f1": f1_vi_per_class, "representation": "V-I image"}),
        pd.DataFrame({"appliance": classes, "f1": f1_decomposed_per_class, "representation": "Decomposed distance"}),
    ]
)
appliance_order = [classes[i] for i in np.argsort(f1_vi_per_class)]
per_class_df["appliance"] = pd.Categorical(per_class_df["appliance"], categories=appliance_order, ordered=True)

per_class_plot = (
    ggplot(per_class_df, aes("appliance", "f1", fill="representation"))
    + geom_bar(stat="identity", position="dodge")
    + coord_flip()
    + scale_fill_manual(values=[BLUES_SEQUENTIAL[2], BRAND_PALETTE[0]])
    + labs(
        title="Per-appliance F1: V-I vs decomposed distance",
        subtitle="PLAID multi-label, sorted by V-I score",
        x="",
        y="F1 (%)",
    )
    + modern_theme(legend_pos="bottom")
    + ggsize(650, 420)
)
per_class_plot

## Why the coffee maker specifically?

CoffeeMaker's weak score is asserted above as a resistive-heating-element
problem, not shown. Restricting to the 648 single-appliance activations
(so each point is one appliance's own current, not several summed
together), the Fryze split gives a direct per-sample measurement: how much
of an appliance's own current is non-active, as a share of its total RMS
current. A purely resistive load carries almost no non-active current by
definition; a motor or switching power supply carries much more.

In [ ]:
single_idx = np.where(n_active == 1)[0]
single_appliance = [classes[int(y[idx].argmax())] for idx in single_idx]

non_active_share = []
for idx in single_idx:
    i_raw = current[idx].astype(np.float64)
    v_raw = voltage[idx].astype(np.float64)
    _, i_non_active = fryze_power_decomposition(i_raw, v_raw)
    share = np.sqrt((i_non_active**2).mean()) / np.sqrt((i_raw**2).mean())
    non_active_share.append(share * 100)

signature_df = pd.DataFrame({"appliance": single_appliance, "non_active_share": non_active_share})
signature_df["appliance"] = pd.Categorical(signature_df["appliance"], categories=appliance_order, ordered=True)

summary_df = signature_df.groupby("appliance", observed=True)["non_active_share"].median().reset_index()

signature_plot = (
    ggplot()
    + geom_jitter(
        aes("appliance", "non_active_share"),
        data=signature_df,
        width=0.15,
        alpha=0.4,
        color=BLUES_SEQUENTIAL[3],
        size=3,
    )
    + geom_point(aes("appliance", "non_active_share"), data=summary_df, color=BRAND_PALETTE[0], size=5, shape=18)
    + coord_flip()
    + labs(
        title="Purely resistive appliances overlap near zero",
        subtitle="Non-active current share, single-appliance activations",
        x="",
        y="Non-active share (%)",
    )
    + modern_theme(legend_pos="none")
    + ggsize(650, 420)
)
signature_plot

## What kind of mistake is it?

A multi-label miss is not one thing. Missing every active appliance is a
different failure than mislabeling one of three. Following the same error
taxonomy the original paper used: *zero-error* (predicts nothing active when
something is), *one-to-one* (gets the single active appliance wrong when
only one is running), and *many-to-many* (confuses at least one appliance
when several are active, split further into how many of them it got wrong).

In [ ]:
def classify_errors(y_true: np.ndarray, y_pred: np.ndarray) -> pd.DataFrame:
    """Categorize every incorrect multi-label prediction by error type.

    Args:
        y_true: Multi-hot ground truth, shape (n_samples, n_appliances).
        y_pred: Multi-hot predictions, same shape.

    Returns:
        DataFrame with one row per *incorrect* sample: `true_count` (how many
        appliances were actually active), `wrong_count` (how many of those
        the model missed or swapped), and `error_type` (one of "zero-error",
        "one-to-one", "single-error", "double-error", "complete-error").
    """
    rows = []
    for true_row, pred_row in zip(y_true, y_pred, strict=True):
        if np.array_equal(true_row, pred_row):
            continue
        true_count = int(true_row.sum())
        correctly_identified = int((true_row * pred_row).sum())
        wrong_count = true_count - correctly_identified

        if true_count >= 1 and pred_row.sum() == 0:
            error_type = "zero-error"
        elif true_count == 1:
            error_type = "one-to-one"
        elif wrong_count == true_count:
            error_type = "complete-error"
        elif wrong_count == 1:
            error_type = "single-error"
        else:
            error_type = "double-error"

        rows.append({"true_count": true_count, "wrong_count": wrong_count, "error_type": error_type})
    return pd.DataFrame(rows)


errors_df = classify_errors(decomposed_distance_result["y_test"], decomposed_distance_result["y_pred"])
error_counts = (
    errors_df["error_type"]
    .value_counts()
    .reindex(["zero-error", "one-to-one", "single-error", "double-error", "complete-error"], fill_value=0)
)

error_plot_df = pd.DataFrame({"error_type": error_counts.index, "count": error_counts.to_numpy()})
error_plot_df["error_type"] = pd.Categorical(error_plot_df["error_type"], categories=error_counts.index, ordered=True)

error_plot = (
    ggplot(error_plot_df, aes("error_type", "count"))
    + geom_bar(stat="identity", fill=BRAND_PALETTE[0], width=0.6)
    + geom_text(aes(label="count"), va="bottom", nudge_y=0.3, size=8)
    + labs(
        title="Most mistakes involve at least one correct appliance",
        subtitle="Error type, decomposed distance, held-out test set",
        x="",
        y="Number of incorrect samples",
    )
    + modern_theme(x_axis_angle=15)
    + ggsize(650, 340)
)
error_plot